# NB14b — Cutoff Calibration / Constraint-Discovery (R-14 Fix)

**Datum:** 2026-05-27
**Scope:** Calibration-Layer NUR — kein neuer Modell-Vergleich, kein TF-Vergleich.
**Trigger:** R-14 (HANDOFF Section 16a) — Standard- und High-Tier kollabieren auf identische Cutoffs auf 5m. Profile-Konzept löchrig.

## Mission
Finde Tier-Cutoffs für das 5m-FX-Modell (V1), die alle 4 Constraints **gleichzeitig** erfüllen:

1. **PF pro Tier (in-sample TEST):** Aggressive ≥ 1.3, Balanced ≥ 1.5, Conservative ≥ 1.8
2. **Signals/Tag/Symbol:** Aggressive 15–30, Balanced 5–10, Conservative 1–4
3. **Cutoff-Separation:** Adjacent Tiers ≥ 0.005 in Probability-Space
4. **Cross-Asset-Stabilität:** PF pro Tier auf jedem Hold-Out-Symbol innerhalb ±20% des Mittels

## Strategien (drei parallel)

1. **Linear-Quantile** (Baseline) — `np.quantile(proba_val, p)` direkt auf probabilities
2. **Logit-Space-Quantile** — entzerrt heavy-tailed Verteilung via `logit(proba)`
3. **Density-Target-Solver** — sucht Cutoffs sodass signals/day-Targets exakt getroffen werden

**Auswahllogik:** Nicht 'höchste PF gewinnt', sondern 'alle 4 Constraints konsistent erfüllt'. Wenn keine Strategie alle erfüllt: explizite Diagnose welcher Constraint failt → fundierte Decision.

**Nicht in Scope von NB14b:**
- Kein Re-Train mit anderen Hyperparametern (R-15, V1.5)
- Keine Feature-Änderungen
- Keine TFs außer 5m
- Keine Asset-Klassen außer FX

## Section 0 — Setup + Drive Mount + Module Sync

In [ ]:
import os, sys, subprocess, gc, json, importlib
from pathlib import Path
from datetime import datetime, timezone

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    os.chdir(PROJECT_ROOT)
    subprocess.run(['rm', '-rf', '/tmp/pace-algo'])
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/ecoNC/pace-algo.git',
                    '/tmp/pace-algo'], check=True)
    subprocess.run(f'mkdir -p {PROJECT_ROOT}/core/eval {PROJECT_ROOT}/core/analysis {PROJECT_ROOT}/core/router',
                   shell=True)
    subprocess.run(f'cp -rf /tmp/pace-algo/core/. {PROJECT_ROOT}/core/', shell=True)
    subprocess.run(f"find {PROJECT_ROOT}/core -type d -name __pycache__ -exec rm -rf {{}} +", shell=True)
    print('Core synced.')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
importlib.invalidate_caches()

RANDOM_SEED = 42
import numpy as np
np.random.seed(RANDOM_SEED)

RUN_DATE      = datetime.now(timezone.utc).strftime('%Y-%m-%d')
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
try:
    GIT_COMMIT = subprocess.check_output(['git', '-C', PROJECT_ROOT, 'rev-parse', '--short', 'HEAD'],
                                          text=True).strip()
except Exception:
    GIT_COMMIT = 'unknown'
EXPERIMENT_ID = f'nb14b_{RUN_TIMESTAMP}_{GIT_COMMIT}'
print(f'EXPERIMENT_ID: {EXPERIMENT_ID}')

In [ ]:
if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'lightgbm>=4.3', 'pyarrow>=15.0'], capture_output=True)

import pandas as pd
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

from core import config as cfg
from core.config import (
    FX_TRAIN_SYMBOLS, FX_HOLDOUT_SYMBOLS,
    TRAIN_END, VAL_END,
)
from core.train.dataset import walk_forward_split, binary_label_for_long, NON_FEATURE_COLS
from core.train.lgbm_trainer import train_lgbm, trading_metrics_from_predictions
from core.analysis.product_metrics import signals_per_day, TF_BARS_PER_DAY

# Konfiguration für NB14b — bewusst klein gehalten
TF = '5m'
R_VALUE = 1.5
WIN_R, LOSS_R = R_VALUE, 1.0
ALL_SYMBOLS = FX_TRAIN_SYMBOLS + FX_HOLDOUT_SYMBOLS

# Constraint-Schwellen (Nico-Lock)
CONSTRAINTS = {
    'pf_aggressive_min':       1.3,
    'pf_balanced_min':         1.5,
    'pf_conservative_min':     1.8,
    'sigs_aggressive_range':   (15, 30),
    'sigs_balanced_range':     (5, 10),
    'sigs_conservative_range': (1, 4),
    'min_cutoff_separation':   0.005,
    'cross_asset_tolerance':   0.20,    # ±20% vom Mean
}

OUTPUT_DIR = Path(PROJECT_ROOT) / 'results' / 'nb14b'
for sub in ('metrics', 'summaries', 'config_snapshots'):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f'TF: {TF}  |  Symbols: {ALL_SYMBOLS}')
print(f'Output: {OUTPUT_DIR}')

## Section 1 — Load Extended Features + Train Model

Assumiert dass NB14 die `_extended.parquet`-Files bereits gebaut hat. Wenn nicht: NB14 Section 2 erst laufen lassen.

In [ ]:
if IN_COLAB:
    DATA_EXT = Path('/content/processed_v2')
    DATA_PROCESSED_LOCAL = Path('/content/processed')
    EXT_DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed_v2'
    DRIVE_BACKUP_PROCESSED = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    DATA_EXT.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED_LOCAL.mkdir(parents=True, exist_ok=True)
    # Restore from Drive backup if local empty
    if len(list(DATA_PROCESSED_LOCAL.glob('labels_*.parquet'))) == 0 and DRIVE_BACKUP_PROCESSED.exists():
        subprocess.run(['rsync', '-ah', f'{DRIVE_BACKUP_PROCESSED}/', f'{DATA_PROCESSED_LOCAL}/'])
    if len(list(DATA_EXT.glob('*_extended.parquet'))) == 0 and EXT_DRIVE_BACKUP.exists():
        subprocess.run(['rsync', '-ah', f'{EXT_DRIVE_BACKUP}/', f'{DATA_EXT}/'])
else:
    DATA_EXT = cfg.DATA_PROCESSED.parent / 'processed_v2'
    DATA_PROCESSED_LOCAL = cfg.DATA_PROCESSED

def load_ext(sym, tf):
    p = DATA_EXT / f'{sym}_{tf}_extended.parquet'
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    # Backwards-compat: legacy NB13 files lack hit_bar_offset
    if 'hit_bar_offset' not in df.columns:
        lp = DATA_PROCESSED_LOCAL / f'labels_{sym}_{tf}_R{R_VALUE}.parquet'
        if lp.exists():
            labels = pd.read_parquet(lp)
            if 'hit_bar_offset' in labels.columns:
                df = df.join(labels[['hit_bar_offset']], how='left')
        if 'hit_bar_offset' not in df.columns:
            df['hit_bar_offset'] = 24
        df['hit_bar_offset'] = df['hit_bar_offset'].fillna(24).astype('int32')
    return df

# Verify all symbols available
missing = [s for s in ALL_SYMBOLS if load_ext(s, TF) is None or load_ext(s, TF).empty]
if missing:
    raise SystemExit(f'Missing _extended.parquet for: {missing} — run NB14 Section 2 first.')
print('Alle Extended-Files vorhanden.')

In [ ]:
# Feature columns (NB11-Winner-27)
probe = load_ext(FX_TRAIN_SYMBOLS[0], TF)
all_cols = [c for c in probe.columns if c not in (NON_FEATURE_COLS | {'hit_bar_offset', 'symbol', 'timeframe'})]
FEATURE_COLS = [
    'hour_sin', 'dist_to_swing_low_atr', 'htf_1h_rsi_14', 'realized_vol_20',
    'atr_percentile_100', 'atr_pct', 'dist_to_swing_high_atr', 'volume_z_score',
    'ema_20_slope_atr', 'hour_cos', 'momentum_composite', 'rvol_20',
    'adx_14', 'ema_20_dist_atr', 'htf_1h_atr_percentile_100',
    'htf_ltf_agree_bull', 'htf_ltf_agree_bear', 'htf_ltf_counter_trend',
    'htf_ltf_alignment_score', 'ltf_rsi_minus_htf_rsi',
    'both_rsi_oversold', 'both_rsi_overbought', 'vol_pct_diff_htf',
    'both_high_vol', 'both_low_vol', 'pullback_in_bull', 'pullback_in_bear',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in all_cols]
del probe
gc.collect()

# Stack train pool (FX_TRAIN_SYMBOLS only) for training
frames = []
for s in FX_TRAIN_SYMBOLS:
    d = load_ext(s, TF)
    if d is not None and not d.empty:
        float_cols = d.select_dtypes(include=['float64']).columns
        if len(float_cols) > 0:
            d = d.astype({c: 'float32' for c in float_cols}, copy=False)
        frames.append(d)
train_pool = pd.concat(frames, axis=0).sort_index().dropna(subset=FEATURE_COLS + ['label'])
del frames
gc.collect()

train_df, val_df, test_df = walk_forward_split(train_pool, TRAIN_END, VAL_END)
del train_pool
gc.collect()
print(f'Train rows: {len(train_df):,}  |  VAL: {len(val_df):,}  |  TEST: {len(test_df):,}')

In [ ]:
# Train LGBM (identische Hyperparams wie NB14)
X_train = train_df[FEATURE_COLS].values.astype(np.float32)
y_train = binary_label_for_long(train_df['label']).values
X_val = val_df[FEATURE_COLS].values.astype(np.float32)
y_val = binary_label_for_long(val_df['label']).values

model = train_lgbm(
    pd.DataFrame(X_train, columns=FEATURE_COLS), pd.Series(y_train),
    pd.DataFrame(X_val, columns=FEATURE_COLS), pd.Series(y_val),
)

proba_val = model.predict(X_val)
X_test = test_df[FEATURE_COLS].values.astype(np.float32)
proba_test = model.predict(X_test)
print(f'Trained LGBM. VAL proba range: [{proba_val.min():.4f}, {proba_val.max():.4f}]  '
      f'mean: {proba_val.mean():.4f}  std: {proba_val.std():.4f}')
print(f'TEST proba range:              [{proba_test.min():.4f}, {proba_test.max():.4f}]  '
      f'mean: {proba_test.mean():.4f}  std: {proba_test.std():.4f}')

## Section 2 — Cutoff-Strategien definieren

Drei Funktionen die je 3 Cutoffs (aggressive/balanced/conservative) aus VAL-Probabilities ableiten.

In [ ]:
from scipy.special import logit as scipy_logit, expit as scipy_expit

def _safe_logit(p, eps=1e-6):
    return np.log(np.clip(p, eps, 1 - eps) / (1 - np.clip(p, eps, 1 - eps)))

def strategy_linear_quantile(proba_val, **kwargs):
    """Baseline: directly use quantiles in probability space.
    Aggressive = top 10%, Balanced = top 3%, Conservative = top 1%."""
    return {
        'aggressive':   float(np.quantile(proba_val, 0.90)),
        'balanced':     float(np.quantile(proba_val, 0.97)),
        'conservative': float(np.quantile(proba_val, 0.99)),
    }

def strategy_logit_quantile(proba_val, **kwargs):
    """Quantile im logit-space (decompressed heavy-tailed distribution)."""
    logit_val = _safe_logit(proba_val)
    return {
        'aggressive':   float(scipy_expit(np.quantile(logit_val, 0.90))),
        'balanced':     float(scipy_expit(np.quantile(logit_val, 0.97))),
        'conservative': float(scipy_expit(np.quantile(logit_val, 0.99))),
    }

def strategy_density_target(proba_val, tf, n_symbols, target_sigs_per_day, n_val_bars, **kwargs):
    """Solve for cutoffs that match target signals/day/symbol on the VAL set.
    
    target_sigs_per_day: dict tier -> sigs/day/symbol target
    """
    bars_per_day = TF_BARS_PER_DAY.get(tf, 288)
    val_days = n_val_bars / (bars_per_day * n_symbols)
    cutoffs = {}
    for tier, target in target_sigs_per_day.items():
        target_n_signals = target * val_days * n_symbols
        target_density = target_n_signals / n_val_bars
        target_density = max(0.0005, min(0.5, target_density))   # clamp
        cutoffs[tier] = float(np.quantile(proba_val, 1.0 - target_density))
    return cutoffs

# Sigs/Day Mid-Range Targets (Mitte der Constraints)
DENSITY_TARGETS = {
    'aggressive':   22.5,    # mid of (15, 30)
    'balanced':     7.5,     # mid of (5, 10)
    'conservative': 2.5,     # mid of (1, 4)
}

print('Strategien definiert. Cutoffs werden aus VAL abgeleitet:')
for name, fn in [('linear', strategy_linear_quantile),
                  ('logit', strategy_logit_quantile)]:
    c = fn(proba_val)
    print(f'  {name:8s}: agg={c["aggressive"]:.4f}  bal={c["balanced"]:.4f}  con={c["conservative"]:.4f}')
c = strategy_density_target(proba_val, TF, len(FX_TRAIN_SYMBOLS), DENSITY_TARGETS, len(val_df))
print(f'  density : agg={c["aggressive"]:.4f}  bal={c["balanced"]:.4f}  con={c["conservative"]:.4f}')

## Section 3 — Apply Strategien auf TEST (FX_TRAIN-Pool)

Pro Strategie: PF, WR, n_trades, sigs/day pro Tier auf in-sample TEST.

In [ ]:
def evaluate_tier(proba, labels_triple, cutoff, R, tf, n_symbols, n_bars):
    mask = proba >= cutoff
    n_sig = int(mask.sum())
    if n_sig == 0:
        return {'n_trades': 0, 'wins': 0, 'losses': 0, 'pf': 0.0, 'wr': 0.0,
                'sigs_per_day': 0.0, 'cutoff': cutoff}
    labs = labels_triple[mask]
    wins = int((labs == 1).sum())
    losses = int((labs == -1).sum())
    pf = (wins * R) / losses if losses > 0 else (float('inf') if wins > 0 else 0.0)
    wr = wins / (wins + losses) if (wins + losses) > 0 else 0.0
    sigs_pd = signals_per_day(n_sig, n_bars, tf, n_symbols)
    return {'n_trades': n_sig, 'wins': wins, 'losses': losses,
            'pf': pf, 'wr': wr, 'sigs_per_day': sigs_pd, 'cutoff': cutoff}

STRATEGIES = {
    'linear':  lambda pv: strategy_linear_quantile(pv),
    'logit':   lambda pv: strategy_logit_quantile(pv),
    'density': lambda pv: strategy_density_target(pv, TF, len(FX_TRAIN_SYMBOLS),
                                                   DENSITY_TARGETS, len(val_df)),
}

test_labels = test_df['label'].values
rows = []
all_strategy_cutoffs = {}
for strat_name, strat_fn in STRATEGIES.items():
    cutoffs = strat_fn(proba_val)
    all_strategy_cutoffs[strat_name] = cutoffs
    for tier in ('aggressive', 'balanced', 'conservative'):
        m = evaluate_tier(proba_test, test_labels, cutoffs[tier], R_VALUE,
                          TF, len(FX_TRAIN_SYMBOLS), len(test_df))
        rows.append({'strategy': strat_name, 'tier': tier, **m})

in_sample_df = pd.DataFrame(rows).round(4)
in_sample_df

## Section 4 — Cross-Asset-Test (Hold-Out Symbols)

Wende Cutoffs jeder Strategie auf GBPUSD, AUDUSD, USDCHF einzeln an.

In [ ]:
holdout_rows = []
for sym in FX_HOLDOUT_SYMBOLS:
    h = load_ext(sym, TF)
    if h is None or h.empty:
        continue
    h = h.dropna(subset=FEATURE_COLS + ['label'])
    _, _, h_test = walk_forward_split(h, TRAIN_END, VAL_END)
    if h_test.empty:
        continue
    X_h = h_test[FEATURE_COLS].values.astype(np.float32)
    proba_h = model.predict(X_h)
    labs_h = h_test['label'].values
    for strat_name, cutoffs in all_strategy_cutoffs.items():
        for tier in ('aggressive', 'balanced', 'conservative'):
            m = evaluate_tier(proba_h, labs_h, cutoffs[tier], R_VALUE, TF, 1, len(h_test))
            holdout_rows.append({'symbol': sym, 'strategy': strat_name, 'tier': tier, **m})
    del h, h_test, X_h, proba_h
    gc.collect()

holdout_df = pd.DataFrame(holdout_rows).round(4)
print('=== Hold-Out per Symbol × Strategy × Tier ===')
pivot_pf = holdout_df.pivot_table(index=['strategy', 'tier'], columns='symbol', values='pf').round(3)
display(pivot_pf)

## Section 5 — Constraint-Solver

Für jede Strategie: prüfe alle 4 Constraints. Eine Strategie 'gewinnt' nur wenn ALLE 4 erfüllt sind.

In [ ]:
def evaluate_strategy(strat_name, in_sample_df, holdout_df, cutoffs, constraints):
    """Returns dict mit pass/fail pro Constraint + verdict."""
    sub = in_sample_df[in_sample_df['strategy'] == strat_name].set_index('tier')
    holdout_sub = holdout_df[holdout_df['strategy'] == strat_name]
    
    checks = {}
    details = {}
    
    # Constraint 1: PF pro Tier (in-sample)
    pf_a = sub.loc['aggressive',   'pf'] if 'aggressive'   in sub.index else 0
    pf_b = sub.loc['balanced',     'pf'] if 'balanced'     in sub.index else 0
    pf_c = sub.loc['conservative', 'pf'] if 'conservative' in sub.index else 0
    checks['pf_aggressive']   = pf_a >= constraints['pf_aggressive_min']
    checks['pf_balanced']     = pf_b >= constraints['pf_balanced_min']
    checks['pf_conservative'] = pf_c >= constraints['pf_conservative_min']
    details['pf'] = {'aggressive': pf_a, 'balanced': pf_b, 'conservative': pf_c}
    
    # Constraint 2: Signals/Day pro Tier
    sigs_a = sub.loc['aggressive',   'sigs_per_day'] if 'aggressive' in sub.index else 0
    sigs_b = sub.loc['balanced',     'sigs_per_day'] if 'balanced' in sub.index else 0
    sigs_c = sub.loc['conservative', 'sigs_per_day'] if 'conservative' in sub.index else 0
    ar = constraints['sigs_aggressive_range']
    br = constraints['sigs_balanced_range']
    cr = constraints['sigs_conservative_range']
    checks['sigs_aggressive']   = ar[0] <= sigs_a <= ar[1]
    checks['sigs_balanced']     = br[0] <= sigs_b <= br[1]
    checks['sigs_conservative'] = cr[0] <= sigs_c <= cr[1]
    details['sigs_per_day'] = {'aggressive': sigs_a, 'balanced': sigs_b, 'conservative': sigs_c}
    
    # Constraint 3: Cutoff-Separation
    c = cutoffs
    gap_ab = abs(c['balanced'] - c['aggressive'])
    gap_bc = abs(c['conservative'] - c['balanced'])
    sep_min = constraints['min_cutoff_separation']
    checks['separation_aggressive_balanced'] = gap_ab >= sep_min
    checks['separation_balanced_conservative'] = gap_bc >= sep_min
    details['cutoff_gaps'] = {'agg→bal': gap_ab, 'bal→con': gap_bc}
    
    # Constraint 4: Cross-Asset-Stabilität (PF pro Tier ±20% vom Mean)
    tol = constraints['cross_asset_tolerance']
    cross_asset_results = {}
    for tier in ('aggressive', 'balanced', 'conservative'):
        pfs = holdout_sub[holdout_sub['tier'] == tier]['pf'].replace([np.inf, -np.inf], np.nan).dropna()
        if len(pfs) == 0:
            checks[f'cross_asset_{tier}'] = False
            continue
        m = pfs.mean()
        # PF wird relativ zum Mean geprüft
        if m > 0:
            max_deviation = max(abs(pfs - m) / m)
            checks[f'cross_asset_{tier}'] = max_deviation <= tol
            cross_asset_results[tier] = {'mean_pf': float(m), 'max_dev_pct': float(max_deviation)}
        else:
            checks[f'cross_asset_{tier}'] = False
            cross_asset_results[tier] = {'mean_pf': float(m), 'max_dev_pct': float('inf')}
    details['cross_asset'] = cross_asset_results
    
    n_pass = sum(1 for v in checks.values() if v)
    n_total = len(checks)
    verdict = 'all_passed' if n_pass == n_total else ('partial' if n_pass >= n_total - 2 else 'failed')
    return {'verdict': verdict, 'n_pass': n_pass, 'n_total': n_total,
            'checks': checks, 'details': details, 'cutoffs': cutoffs}

strategy_results = {}
for strat_name in STRATEGIES.keys():
    strategy_results[strat_name] = evaluate_strategy(
        strat_name, in_sample_df, holdout_df, all_strategy_cutoffs[strat_name], CONSTRAINTS
    )
    print(f'\n--- Strategy: {strat_name.upper()} ---')
    r = strategy_results[strat_name]
    print(f'  Verdict: {r["verdict"]}  ({r["n_pass"]}/{r["n_total"]} passed)')
    for k, v in r['checks'].items():
        sym = '✓' if v else '✗'
        print(f'    {sym} {k}')

## Section 6 — Auswahl + Final Cutoffs

In [ ]:
# Pick winner: all_passed > most passes; if tie, prefer density > logit > linear
preference_order = ['density', 'logit', 'linear']
winners = []
for strat in preference_order:
    r = strategy_results[strat]
    winners.append((strat, r['verdict'], r['n_pass']))

# Sortiere: 1) all_passed, 2) max n_pass, 3) preference
winner = max(winners, key=lambda x: (x[1] == 'all_passed', x[2], -preference_order.index(x[0])))
winner_name = winner[0]
winner_result = strategy_results[winner_name]

print('=' * 70)
print(f'WINNER STRATEGY: {winner_name.upper()}')
print(f'Verdict: {winner_result["verdict"]}  ({winner_result["n_pass"]}/{winner_result["n_total"]} constraints passed)')
print('=' * 70)
print(f'\nFINAL V1 CUTOFFS (5m FX, VAL-derived):')
for tier in ('aggressive', 'balanced', 'conservative'):
    c = winner_result['cutoffs'][tier]
    pf = winner_result['details']['pf'][tier]
    sigs = winner_result['details']['sigs_per_day'][tier]
    print(f'  {tier:13s}: cutoff={c:.6f}  PF={pf:.3f}  sigs/day={sigs:.2f}')

if winner_result['verdict'] != 'all_passed':
    print(f'\n⚠️  WINNER ist NICHT all_passed. Failing Constraints:')
    for k, v in winner_result['checks'].items():
        if not v:
            print(f'    ✗ {k}')

## Section 7 — Per-Symbol Calibration (Winner-Strategie)

Wenn die Cross-Asset-Stabilität failt, kalibrieren wir per Symbol — gleiche Strategie, eigene Cutoffs pro Symbol.

In [ ]:
per_symbol_rows = []
for sym in ALL_SYMBOLS:
    h = load_ext(sym, TF)
    if h is None or h.empty:
        continue
    h = h.dropna(subset=FEATURE_COLS + ['label'])
    h_tr, h_va, h_te = walk_forward_split(h, TRAIN_END, VAL_END)
    if h_va.empty or h_te.empty:
        continue
    X_hv = h_va[FEATURE_COLS].values.astype(np.float32)
    X_ht = h_te[FEATURE_COLS].values.astype(np.float32)
    p_hv = model.predict(X_hv)
    p_ht = model.predict(X_ht)
    # Apply winner strategy to this symbol's VAL
    if winner_name == 'linear':
        cuts = strategy_linear_quantile(p_hv)
    elif winner_name == 'logit':
        cuts = strategy_logit_quantile(p_hv)
    else:  # density
        cuts = strategy_density_target(p_hv, TF, 1, DENSITY_TARGETS, len(h_va))
    labs = h_te['label'].values
    for tier in ('aggressive', 'balanced', 'conservative'):
        m = evaluate_tier(p_ht, labs, cuts[tier], R_VALUE, TF, 1, len(h_te))
        per_symbol_rows.append({'symbol': sym, 'tier': tier, **m})
    del h, h_tr, h_va, h_te, X_hv, X_ht, p_hv, p_ht
    gc.collect()

per_symbol_df = pd.DataFrame(per_symbol_rows).round(4)
print(f'=== Per-Symbol Calibration ({winner_name}-Strategie) ===')
pivot_per_sym = per_symbol_df.pivot_table(index='tier', columns='symbol', values=['pf', 'sigs_per_day']).round(3)
display(pivot_per_sym)

## Section 8 — Final Validation des Winners

Sanity-Check: stimmen die Cutoff-Werte mit der Performance auf TEST überein? Und reicht die Cross-Asset-Stabilität?

In [ ]:
print('=' * 70)
print('FINAL VALIDATION SUMMARY')
print('=' * 70)

# Cross-asset stability per tier (per-symbol calibrated)
for tier in ('aggressive', 'balanced', 'conservative'):
    sub = per_symbol_df[per_symbol_df['tier'] == tier]
    pfs = sub['pf'].replace([np.inf, -np.inf], np.nan).dropna()
    sigs = sub['sigs_per_day']
    print(f'\n{tier.upper()}:')
    print(f'  PF per symbol:     mean={pfs.mean():.3f}  min={pfs.min():.3f}  max={pfs.max():.3f}  cv={pfs.std()/pfs.mean():.3f}' if pfs.mean() > 0 else f'  PF per symbol: insufficient data')
    print(f'  sigs/day per sym:  mean={sigs.mean():.2f}  min={sigs.min():.2f}  max={sigs.max():.2f}')
    
# Check if per-symbol calibration improves over global
print(f'\n--- Global ({winner_name}) vs Per-Symbol comparison ---')
global_winner = strategy_results[winner_name]
for tier in ('aggressive', 'balanced', 'conservative'):
    global_pf = global_winner['details']['pf'][tier]
    per_sym_pfs = per_symbol_df[per_symbol_df['tier'] == tier]['pf'].replace([np.inf, -np.inf], np.nan).dropna()
    per_sym_min = per_sym_pfs.min() if len(per_sym_pfs) > 0 else 0
    print(f'  {tier:13s}: global_pf={global_pf:.3f}  per_sym_min={per_sym_min:.3f}  '
          f'{"✓" if per_sym_min > 0 else "!"} all-syms-above-1.0={(per_sym_pfs > 1.0).all() if len(per_sym_pfs) > 0 else False}')

## Section 9 — Result Persistence (results/nb14b/)

In [ ]:
# Persist CSVs
in_sample_df.to_csv(OUTPUT_DIR / 'metrics' / f'in_sample_per_strategy_{RUN_DATE}.csv', index=False)
holdout_df.to_csv(OUTPUT_DIR / 'metrics' / f'holdout_per_strategy_{RUN_DATE}.csv', index=False)
per_symbol_df.to_csv(OUTPUT_DIR / 'metrics' / f'per_symbol_winner_{RUN_DATE}.csv', index=False)

snapshot = {
    'experiment_id':     EXPERIMENT_ID,
    'run_date':          RUN_DATE,
    'git_commit':        GIT_COMMIT,
    'tf':                TF,
    'train_symbols':     FX_TRAIN_SYMBOLS,
    'holdout_symbols':   FX_HOLDOUT_SYMBOLS,
    'feature_cols':      FEATURE_COLS,
    'constraints':       CONSTRAINTS,
    'density_targets':   DENSITY_TARGETS,
    'all_strategy_cutoffs': all_strategy_cutoffs,
    'strategy_results':  {k: {kk: vv for kk, vv in v.items() if kk != 'details' or True}
                          for k, v in strategy_results.items()},
    'winner':            winner_name,
    'winner_verdict':    winner_result['verdict'],
    'winner_cutoffs':    winner_result['cutoffs'],
    'per_symbol_metrics': per_symbol_df.to_dict(orient='records'),
}
with open(OUTPUT_DIR / 'summaries' / f'nb14b_full_snapshot_{RUN_DATE}.json', 'w') as f:
    json.dump(snapshot, f, indent=2, default=str)

with open(OUTPUT_DIR / 'config_snapshots' / f'{EXPERIMENT_ID}_config.json', 'w') as f:
    json.dump({
        'experiment_id': EXPERIMENT_ID,
        'tf':            TF,
        'constraints':   CONSTRAINTS,
        'density_targets': DENSITY_TARGETS,
        'strategies':    list(STRATEGIES.keys()),
    }, f, indent=2)

print(f'✅ Results in {OUTPUT_DIR}')

## Section 10 — Auto-Push to GitHub

In [ ]:
import shutil
if not IN_COLAB:
    print('Local run — skip push.')
else:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get('GITHUB_TOKEN') or userdata.get('GH_PAT')
    except Exception:
        GH_TOKEN = None
    if not GH_TOKEN:
        print('⚠️ kein GH-Token in Secrets — Results bleiben im Drive')
    else:
        TMP_REPO = Path('/tmp/pace-algo-push')
        if TMP_REPO.exists():
            shutil.rmtree(TMP_REPO)
        subprocess.run(['git', 'clone', '--quiet',
                        f'https://{GH_TOKEN}@github.com/ecoNC/pace-algo.git', str(TMP_REPO)], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.name', 'ecoNC'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.email',
                        'ecoNC@users.noreply.github.com'], check=True)
        copied = []
        for f in sorted(OUTPUT_DIR.rglob(f'*{RUN_DATE}*')):
            if not f.is_file():
                continue
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        for f in sorted((OUTPUT_DIR / 'config_snapshots').glob(f'*{EXPERIMENT_ID}*')):
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        subprocess.run(['git', '-C', str(TMP_REPO), 'pull', '--rebase', '--quiet', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'add', 'results/'], check=True)
        st = subprocess.run(['git', '-C', str(TMP_REPO), 'status', '--porcelain'], capture_output=True, text=True)
        if not st.stdout.strip():
            print('Nothing to commit.')
        else:
            msg = (f'NB14b cutoff calibration {RUN_DATE} — winner: {winner_name}, '
                   f'verdict: {winner_result["verdict"]}, '
                   f'{winner_result["n_pass"]}/{winner_result["n_total"]} constraints')
            subprocess.run(['git', '-C', str(TMP_REPO), 'commit', '-m', msg], check=True)
            sha = subprocess.run(['git', '-C', str(TMP_REPO), 'rev-parse', '--short', 'HEAD'],
                                  capture_output=True, text=True).stdout.strip()
            subprocess.run(['git', '-C', str(TMP_REPO), 'push', 'origin', 'main'], check=True)
            print(f'✓ Pushed as ecoNC ({sha})')
        shutil.rmtree(TMP_REPO)

## Section 11 — Final Verdict

Nach dem Run analysiert Claude die JSON und:
1. Lockt Winner-Strategie als V1-Cutoff-Mechanik in ANN-011 + model_registry
2. Updated pine_router_design.md mit den finalen Cutoff-Werten
3. Schließt R-14 als RESOLVED
4. Falls KEIN all_passed: re-research statt deploy